# 05 — TEMA validation (short-term 4h book)

Frozen live stack **9 / 90 / 199**, agreement `>= 1`, isolated **10×**, SL 1.5×ATR / TP 2.5×ATR, same-bar both → SL. QMIE stays signal-only. **Do not retune live `W_*`.**

This notebook is the hedge-fund read of *this* book only — not spot, not the BTC/QQQ/GLD Carver trio.

## Protocol

| Slice | Window | Use |
|---|---|---|
| IS | 2019-09-01 → 2022-12-31 | describe, never steal OOS for a story |
| OOS | 2023-01-01 → today | the test |
| Warmup | 220 4h bars | OOS indicators seeded from last 220 IS bars |

Vision USDT-M 4h starts ~2020-01, not 2018.

## How to read equity and drawdown

KPIs are **daily-marked** (`ann=365`). A 4h bar Sharpe with `ann=365` understates vol.

The lab default `$10k account + $100 isolated stake` makes max DD look tiny (~3%). That is **not** control — it is a 1% wallet. This notebook also compounds:

* **1% compounding** — each ticket risks 1% of *current* equity as isolated margin (prop-like).
* **Full isolated wallet** — the whole account is the stake. One 1.5×ATR SL at 10× is a mid-teens hit, not a rounding error.

## H8

Frozen TEMA has a usable OOS path with controlled DD once stake is honest.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent.parent
elif (ROOT / "research").exists():
    pass
elif (ROOT / "python" / "research").exists():
    ROOT = ROOT / "python"
sys.path.insert(0, str(ROOT))
print("python root", ROOT)


In [ ]:
import pandas as pd
from research.trend_lab.evaluate import eval_tema
from research.trend_lab.data import load_symbol
from research.trend_lab.metrics import kpi_table
from research.trend_lab.plots import equity_overlay, price_signals, rolling_sharpe_fig, underwater
from research.trend_lab.protocol import SPLIT, WARMUP_BARS, split_frame
from research.trend_lab.tema_robust import daily_kpis
from research.trend_lab.tema_system import TemaParams, compound_trades, daily_equity, tema_bar_equity

print("IS", SPLIT.is_start, "→", SPLIT.is_end)
print("OOS", SPLIT.oos_start, "→", SPLIT.oos_end)
print("warmup", WARMUP_BARS)
print("frozen 9/90/199 10× isolated — Optuna does not belong in this notebook")


In [ ]:
btc, src = load_symbol("BTCUSDT", "4h")
print("BTC 4h", src, len(btc), btc.index[0], "→", btc.index[-1])
parts = split_frame(btc)
p10 = TemaParams(leverage=10.0)
p1 = TemaParams(leverage=1.0)
t10 = eval_tema(btc, p10)
t1 = eval_tema(btc, p1)
oos_idx, is_idx = parts["oos"].index, parts["is"].index

comp_1pct = compound_trades(t10["oos_trades"], start_eq=10_000.0, risk_frac=0.01, leverage=10.0, cost_bps=p10.cost_bps)
comp_full = compound_trades(t10["oos_trades"], start_eq=10_000.0, risk_frac=1.0, leverage=10.0, cost_bps=p10.cost_bps)

def deq(idx, tr, start=10_000.0):
    return daily_equity(tema_bar_equity(idx, tr, start_eq=start)["equity"])

board = kpi_table({
    "frozen_10x_IS_daily": t10["is_daily"],
    "frozen_10x_OOS_daily": t10["oos_daily"],
    "frozen_1x_OOS_daily": t1["oos_daily"],
    "compound_1pct_OOS": daily_kpis(oos_idx, comp_1pct),
    "compound_full_wallet_OOS": daily_kpis(oos_idx, comp_full),
})
display(board.round(3))
print("IS trades", len(t10["is_trades"]), "OOS trades", len(t10["oos_trades"]))
if len(t10["oos_trades"]):
    display(t10["oos_trades"]["outcome"].value_counts().to_frame("n"))
    display(t10["oos_trades"][["r", "pnl", "bars"]].describe().round(3))


In [ ]:
eq_art = deq(oos_idx, t10["oos_trades"])
eq_1x = deq(oos_idx, t1["oos_trades"])
eq_1pct = deq(oos_idx, comp_1pct)
eq_full = deq(oos_idx, comp_full)
eq_is = deq(is_idx, t10["is_trades"])

equity_overlay({
    "$10k+$100 10× (artifact)": eq_art,
    "1× same trades": eq_1x,
    "1% compounding": eq_1pct,
    "full isolated wallet": eq_full,
}, "OOS TEMA 9/90/199 — daily-marked equity").show()
rolling_sharpe_fig({
    "artifact": eq_art.pct_change().fillna(0),
    "1%": eq_1pct.pct_change().fillna(0),
    "full wallet": eq_full.pct_change().fillna(0),
}, 90, "OOS 90d rolling Sharpe").show()
underwater(eq_art, "OOS DD — $10k+$100 (understated)").show()
underwater(eq_1pct, "OOS DD — 1% compounding").show()
underwater(eq_full, "OOS DD — full isolated wallet").show()
equity_overlay({"IS frozen 10×": eq_is}, "IS TEMA — daily-marked").show()
underwater(eq_is, "IS DD — $10k+$100").show()
price_signals(
    parts["oos"],
    entries=pd.DatetimeIndex(t10["oos_trades"]["entry_time"]) if len(t10["oos_trades"]) else None,
    exits=pd.DatetimeIndex(t10["oos_trades"]["exit_time"]) if len(t10["oos_trades"]) else None,
    title="BTC 4h OOS — frozen TEMA entries",
    max_bars=800,
).show()


## How to read this

If OOS Sharpe on the $100-stake book is ~0.3 and DD is −3%, **do not** call that a 10× edge with tight risk. Leverage scaled expectancy (H2 in notebook 01); it did not invent a Sharpe. The full-wallet curve is the honest “what if this *were* the book.” 1% compounding is the honest “what if we size like a desk.”

0 liquidations on this sample is a KPI, not a guarantee — isolated cap is load-bearing (`pnl >= -stake`).

Promote-to-live still needs DF + OOS vs frozen 9/90/199. This notebook does not search.
